# Audit a small validation set with TorchCAM

[Open in Colab](https://colab.research.google.com/github/frgfm/notebooks/blob/main/torch-cam/validation_audit.ipynb)

Inspect a fixed set of classifier failures **and successful controls**. This notebook saves a predicted-versus-expected explanation bundle per sample and a compact JSON audit with scores, blank-map flags, errors, and optional group counts.

The runnable example uses four generated images and a tiny classifier with explicit weights. Labels describe the central square's colour; the classifier averages colour over the whole image. This makes two correct and two incorrect predictions without training, external data, or weight downloads. These are controlled examples, not evidence about a real dataset or model bias.

Replace the model, labels, preprocessing, and sample list with the owner's existing inference path. The audit does not retrain a model, diagnose a proxy automatically, or estimate full-dataset accuracy.


## 1. Install the tested implementation

Use Python 3.11 or newer. This snapshot includes model-view size validation and caller-supplied bundle context after TorchCAM 0.5.0. Runtime dependency versions are reported below. The CPU example takes no training steps.


In [ ]:
%pip install "torchcam[scripts] @ git+https://github.com/frgfm/torch-cam.git@0be00ce14a487555e3f89e21b0be6f7106c51e1e"

In [ ]:
import json
import platform
import tempfile
from importlib.metadata import version
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from torch import nn
from torchcam.explain import explain
from torchvision.transforms.functional import to_tensor

SOURCE_REVISION = "0be00ce14a487555e3f89e21b0be6f7106c51e1e"
torch.set_num_threads(2)
torch.manual_seed(0)
runtime_versions = {
    "python": platform.python_version(),
    **{package: version(package) for package in ("torch", "torchvision", "torchcam", "Pillow")},
}
print(json.dumps({"source_revision": SOURCE_REVISION, **runtime_versions}, indent=2))

## 2. Reuse the model and preprocessing

This small fixture makes the workflow runnable. It uses an identity colour projection and global averaging; it has no learned checkpoint. The stable checkpoint identifier describes those explicit weights.

For a real audit, replace this cell with the trusted checkpoint loader, ordered labels, and evaluation preprocessing from your repository. Return a **model-view PIL image** after the same resize/crop as the tensor, before normalization. Do not stretch a crop-based CAM over the untouched original. Replace the context identifiers with the owner's records, preferably an immutable checkpoint revision or checksum.


In [ ]:
class DemoColorClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Conv2d(3, 2, 1, bias=False)
        with torch.no_grad():
            self.features.weight.zero_()
            self.features.weight[0, 0, 0, 0] = 1
            self.features.weight[1, 2, 0, 0] = 1

    def forward(self, inputs):
        return self.features(inputs).mean(dim=(2, 3))


model = DemoColorClassifier().eval()
class_names = ("red", "blue")
target_layer = "features"
run_context = {
    "checkpoint_id": "demo-colour-projection-v1",
    "preprocessing_id": "RGB; unchanged spatial dimensions; pixels scaled by 1/255",
    "split_id": "fixed-demo-probes-v1",
    "torchcam_source_revision": SOURCE_REVISION,
}


def prepare_input(image):
    model_image = image.convert("RGB")
    return to_tensor(model_image).unsqueeze(0), model_image

## 3. Choose fixed probes before inspecting the CAMs

Keep both mistakes and successful controls. Record each sample's expected class from trusted annotations, not from model predictions. Optional group IDs can describe a camera, source, acquisition site, or another known attribute.

Here the groups are the central square's size. They are fixture labels, not demographic or deployment domains. A hand-picked probe set supports diagnosis; its accuracy is not a representative population estimate.


In [ ]:
workspace = Path(tempfile.mkdtemp(prefix="torchcam-validation-"))
data_dir = workspace / "samples"
data_dir.mkdir()
samples = []
for class_idx, colour, background in ((0, "red", "blue"), (1, "blue", "red")):
    for size, group_id in ((6, "large-square"), (2, "small-square")):
        sample_id = f"{colour}-{group_id}"
        image = Image.new("RGB", (8, 8), background)
        offset = (8 - size) // 2
        ImageDraw.Draw(image).rectangle((offset, offset, offset + size - 1, offset + size - 1), fill=colour)
        image_path = data_dir / f"{sample_id}.png"
        image.save(image_path)
        samples.append(
            {
                "sample_id": sample_id,
                "image_path": image_path,
                "expected_class_idx": class_idx,
                "group_id": group_id,
            }
        )

## 4. Explain each probe and preserve failures

The loop reuses the single-image API and the configuration above. IDs are records, never output paths. Duplicate/empty IDs and invalid labels are rejected before writing. Each sample gets a numbered directory.

An extraction error stays in the report with its exception type and message. A finite constant map is flagged as blank; it is not evidence that a visual feature is absent. Correctness is computed only when explanation succeeds. The report always shows explained and error counts beside that conditional accuracy.

The output directory must be new. The audit JSON is written last: without it, the audit is incomplete. A report containing error rows is complete as a record of attempts, not a clean bill of health.


In [ ]:
def summarize(rows):
    explained = [row for row in rows if row["status"] == "ok"]
    correct = sum(row["correct"] for row in explained)
    return {
        "attempted": len(rows),
        "explained": len(explained),
        "errors": len(rows) - len(explained),
        "correct": correct,
        "incorrect": len(explained) - correct,
        "blank_samples": sum(bool(row["blank_maps"]) for row in explained),
        "accuracy_on_explained_samples": correct / len(explained) if explained else None,
    }


def audit_samples(samples, output_dir):
    samples = list(samples)
    if not samples:
        raise ValueError("Choose at least one fixed probe.")
    required = {"sample_id", "image_path", "expected_class_idx"}
    for sample in samples:
        if not required <= sample.keys():
            raise ValueError("Each probe needs a sample ID, image path, and expected class.")
        if not isinstance(sample["sample_id"], str) or not sample["sample_id"].strip():
            raise ValueError("Sample IDs must be nonempty strings.")
        expected = sample["expected_class_idx"]
        if isinstance(expected, bool) or not isinstance(expected, int) or not 0 <= expected < len(class_names):
            raise ValueError("Expected classes must use the model's ordered label indices.")
        if sample.get("group_id") is not None and not isinstance(sample["group_id"], str):
            raise ValueError("Group IDs must be strings or omitted.")
    if len({sample["sample_id"] for sample in samples}) != len(samples):
        raise ValueError("Sample IDs must be unique.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True)  # Refuse to overwrite an earlier audit.
    rows = []
    for index, sample in enumerate(samples):
        expected = sample["expected_class_idx"]
        row = {
            "sample_id": sample["sample_id"],
            "image_path": str(sample["image_path"]),
            "group_id": sample.get("group_id"),
            "expected_class_idx": expected,
            "expected_class_name": class_names[expected],
        }
        try:
            with Image.open(sample["image_path"]) as source:
                input_tensor, model_image = prepare_input(source)
            result = explain(
                model,
                input_tensor,
                expected_class_idx=expected,
                class_names=class_names,
                target_layer=target_layer,
            )
            context = {**run_context, "sample_id": sample["sample_id"]}
            if sample.get("group_id") is not None:
                context["group_id"] = sample["group_id"]
            bundle = result.save(output_dir / f"sample-{index:03d}", model_image, context=context)
            probabilities = result.logits.softmax(dim=1)[0]
            predicted = result.predicted_class_idx
            row.update(
                {
                    "status": "ok",
                    "predicted_class_idx": predicted,
                    "predicted_class_name": class_names[predicted],
                    "correct": predicted == expected,
                    "predicted_logit": result.logits[0, predicted].item(),
                    "expected_logit": result.logits[0, expected].item(),
                    "predicted_probability": probabilities[predicted].item(),
                    "expected_probability": probabilities[expected].item(),
                    "bundle": str(bundle.relative_to(output_dir)),
                    "blank_maps": [
                        {"class_idx": class_idx, "layer_idx": layer_idx}
                        for class_idx, maps in result.cams.items()
                        for layer_idx, cam in enumerate(maps)
                        if cam.min().item() == cam.max().item()
                    ],
                }
            )
        except (OSError, RuntimeError, TypeError, ValueError) as error:
            row.update({"status": "error", "error_type": type(error).__name__, "error": str(error)})
        rows.append(row)

    report = {
        "schema_version": 1,
        "scope": "fixed diagnostic probes; not a population accuracy estimate",
        "context": run_context,
        "runtime_versions": runtime_versions,
        "summary": summarize(rows),
        "groups": [
            {"group_id": group_id, **summarize([row for row in rows if row["group_id"] == group_id])}
            for group_id in dict.fromkeys(row["group_id"] for row in rows)
        ],
        "samples": rows,
    }
    (output_dir / "audit.json").write_text(
        json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8"
    )
    return report

In [ ]:
audit_dir = workspace / "audit"
report = audit_samples(samples, audit_dir)
print(json.dumps({"summary": report["summary"], "groups": report["groups"]}, indent=2))
for row in report["samples"]:
    if row["status"] == "ok":
        print(
            f"{row['sample_id']}: expected={row['expected_class_name']}, "
            f"predicted={row['predicted_class_name']} "
            f"({row['predicted_probability']:.1%}), "
            f"blank_maps={len(row['blank_maps'])}, bundle={row['bundle']}"
        )
    else:
        print(f"{row['sample_id']}: ERROR {row['error_type']}: {row['error']}")
print(f"Saved audit and per-sample evidence: {audit_dir}")

## 5. Run the smallest acceptance check

The fixture must report four attempts, two correct predictions, and two mistakes, with no dropped records. We then run **separate fault probes** to verify that a missing image remains an error and a black image's constant maps remain blank. These probes are tests of the reporting contract, not validation labels for a real task.

Inspect the saved model-view image and class maps in each successful bundle. The assertions also check provenance, class references, files, dimensions, and finite map values. No real-world diagnosis or model improvement is claimed by these checks.


In [ ]:
assert report["summary"] == {
    "attempted": 4,
    "explained": 4,
    "errors": 0,
    "correct": 2,
    "incorrect": 2,
    "blank_samples": 0,
    "accuracy_on_explained_samples": 0.5,
}
assert {group["group_id"]: group["correct"] for group in report["groups"]} == {
    "large-square": 2,
    "small-square": 0,
}
assert json.loads((audit_dir / "audit.json").read_text(encoding="utf-8")) == report
for row in report["samples"]:
    bundle = audit_dir / row["bundle"]
    manifest = json.loads((bundle / "manifest.json").read_text(encoding="utf-8"))
    assert manifest["schema_version"] == 1
    assert manifest["context"] == {
        **run_context,
        "sample_id": row["sample_id"],
        "group_id": row["group_id"],
    }
    assert manifest["prediction"]["class_idx"] == row["predicted_class_idx"]
    assert manifest["expected"]["class_idx"] == row["expected_class_idx"]
    assert manifest["image_size"] == [8, 8]
    with Image.open(bundle / manifest["input_image"]) as saved_input:
        with Image.open(row["image_path"]) as original:
            np.testing.assert_array_equal(saved_input, original)
    for class_data in manifest["classes"].values():
        for artifact in class_data["artifacts"]:
            assert all((bundle / artifact[key]).is_file() for key in ("map", "heatmap", "overlay"))
            cam = np.load(bundle / artifact["map"], allow_pickle=False)
            assert cam.ndim == 2 and cam.dtype == np.float32 and np.isfinite(cam).all()
            with Image.open(bundle / artifact["overlay"]) as overlay:
                assert overlay.size == (8, 8)

blank_path = data_dir / "blank.png"
Image.new("RGB", (8, 8)).save(blank_path)
faults = audit_samples(
    [
        {"sample_id": "blank-control", "image_path": blank_path, "expected_class_idx": 0},
        {"sample_id": "missing-control", "image_path": data_dir / "missing.png", "expected_class_idx": 0},
    ],
    workspace / "fault-check",
)
assert faults["summary"]["attempted"] == 2
assert faults["summary"]["explained"] == 1 and faults["summary"]["errors"] == 1
assert faults["summary"]["blank_samples"] == 1
assert faults["samples"][0]["blank_maps"]
assert faults["samples"][1]["error_type"] == "FileNotFoundError"
assert "bundle" not in faults["samples"][1]

for probes, directory, exception in (
    (samples, audit_dir, FileExistsError),
    ([samples[0], samples[0]], workspace / "duplicate-check", ValueError),
):
    try:
        audit_samples(probes, directory)
    except exception:
        pass
    else:
        raise AssertionError("An existing audit or duplicate sample IDs must be rejected.")
assert not (workspace / "duplicate-check").exists()
print("PASS: correct/incorrect counts, groups, evidence, blank maps, explicit errors, and no overwrite.")

## Use the report to choose the next check

On your own validation probes, inspect successful controls alongside mistakes. First verify labels, the real preprocessing, and crop alignment. Use differences in class-associated maps to formulate a specific hypothesis, then test it with independent evidence.

The report contains conditional accuracy for this fixed set, error coverage, and the supplied group counts. Softmax probabilities are not calibrated confidence. Independently normalized CAM brightness cannot compare class probabilities. A blank map, group difference, or attractive overlay does not establish model bias, causal influence, object localization, or a training remedy.

Keep the audit away from the final test split when choosing interventions. This notebook deliberately stops at evidence collection. It does not choose augmentation, rebalance data, or change the model.

For another run, choose a new output directory. Download the audit and its sample bundles before a temporary notebook runtime ends. To audit a real model, replace section 2 and the fixed sample list in section 3; preserve their sample IDs, trusted labels, and available group metadata. The four-sample acceptance counts in section 5 apply only to this fixture.
